In [1]:
!pip -q install geopandas category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.4/87.4 kB 3.0 MB/s eta 0:00:00


In [2]:
import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from category_encoders import TargetEncoder

from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
BASE_PATH = "/content/drive/MyDrive/2026-Summer/IDXExchange-Intern"
DATA_PATH = os.path.join(BASE_PATH, "data")
OUTPUT_PATH = os.path.join(BASE_PATH, "week6_revised_outputs")
SPLIT_PATH = os.path.join(BASE_PATH, "split-data")
DISTRICT_PATH = os.path.join(DATA_PATH, "DistrictAreas2425.shp")

os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(SPLIT_PATH, exist_ok=True)

print("Project:", BASE_PATH)
print("Outputs:", OUTPUT_PATH)
print("District file:", DISTRICT_PATH)

Project: /content/drive/MyDrive/2026-Summer/IDXExchange-Intern
Outputs: /content/drive/MyDrive/2026-Summer/IDXExchange-Intern/week6_revised_outputs
District file: /content/drive/MyDrive/2026-Summer/IDXExchange-Intern/data/DistrictAreas2425.shp


##Load and combine all monthly MLS transaction datasets.

In [5]:
months = [
    "202502", "202503", "202504", "202505",
    "202506", "202507", "202508", "202509",
    "202510", "202511", "202512", "202601",
    "202602", "202603", "202604", "202605", "202606"
]

monthly_frames = []

for month in months:
    file_path = os.path.join(DATA_PATH, f"CRMLSSold{month}.csv")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Monthly file not found: {file_path}")

    month_frame = pd.read_csv(file_path, low_memory=False)
    month_frame["_source_month"] = month
    monthly_frames.append(month_frame)

raw_data = pd.concat(monthly_frames, ignore_index=True)

print("Files loaded:", len(monthly_frames))
print("Combined shape:", raw_data.shape)
print("Source months:", raw_data["_source_month"].nunique())

Files loaded: 17
Combined shape: (369739, 79)
Source months: 17


##Filter the target property type and prepare the required variables.

In [6]:
data = raw_data.loc[
    (raw_data["PropertyType"] == "Residential") &
    (raw_data["PropertySubType"] == "SingleFamilyResidence")
].copy()

data["CloseDate"] = pd.to_datetime(data["CloseDate"], errors="coerce")
data["ClosePrice"] = pd.to_numeric(data["ClosePrice"], errors="coerce")

base_numeric_features = [
    "Latitude",
    "Longitude",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "YearBuilt",
    "Stories",
    "GarageSpaces",
    "ParkingTotal",
    "AssociationFee",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea"
]

base_high_cardinality_features = [
    "City",
    "PostalCode",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict"
]

base_low_cardinality_features = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN"
]

target = "ClosePrice"
date_col = "CloseDate"

required_columns = (
    base_numeric_features
    + base_high_cardinality_features
    + base_low_cardinality_features
    + [target, date_col]
)

missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise KeyError(f"Required source columns are missing: {missing_columns}")

data = data[required_columns].copy()
data = data.dropna(subset=[date_col, target])
data = data.loc[data[target] > 0].copy()

data.loc[pd.to_numeric(data["ParkingTotal"], errors="coerce") < 0, "ParkingTotal"] = np.nan
data.loc[pd.to_numeric(data["LivingArea"], errors="coerce") <= 0, "LivingArea"] = np.nan
data.loc[pd.to_numeric(data["LotSizeSquareFeet"], errors="coerce") < 0, "LotSizeSquareFeet"] = np.nan

print("Modeling rows:", f"{len(data):,}")
print("Date range:", data[date_col].min().date(), "to", data[date_col].max().date())
print("Median ClosePrice:", f"${data[target].median():,.0f}")

Modeling rows: 186,195
Date range: 2025-02-01 to 2026-06-30
Median ClosePrice: $896,000


##Split the dataset into chronological training, validation, and test sets.

In [7]:
train_raw = data.loc[
    (data[date_col] >= "2025-02-01") &
    (data[date_col] < "2026-02-01")
].copy()

validation_raw = data.loc[
    (data[date_col] >= "2026-02-01") &
    (data[date_col] < "2026-06-01")
].copy()

test_raw = data.loc[
    (data[date_col] >= "2026-06-01") &
    (data[date_col] < "2026-07-01")
].copy()

split_summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Rows": [len(train_raw), len(validation_raw), len(test_raw)],
    "Start": [
        train_raw[date_col].min(),
        validation_raw[date_col].min(),
        test_raw[date_col].min()
    ],
    "End": [
        train_raw[date_col].max(),
        validation_raw[date_col].max(),
        test_raw[date_col].max()
    ],
    "Median ClosePrice": [
        train_raw[target].median(),
        validation_raw[target].median(),
        test_raw[target].median()
    ]
})

assert train_raw[date_col].max() < validation_raw[date_col].min()
assert validation_raw[date_col].max() < test_raw[date_col].min()
assert len(train_raw) > 0 and len(validation_raw) > 0 and len(test_raw) > 0

split_summary

,Split,Rows,Start,End,Median ClosePrice
0,Train,129555,2025-02-01,2026-01-31,888000.0
1,Validation,43782,2026-02-01,2026-05-31,905000.0
2,Test,12858,2026-06-01,2026-06-30,925000.0


##Create additional engineered features

In [8]:
def add_property_features(frame):
    result = frame.copy()

    bedrooms = pd.to_numeric(result["BedroomsTotal"], errors="coerce")
    bathrooms = pd.to_numeric(result["BathroomsTotalInteger"], errors="coerce")
    year_built = pd.to_numeric(result["YearBuilt"], errors="coerce")

    result["BedBathRatio"] = bedrooms / bathrooms.replace(0, np.nan)
    result["PropertyAge"] = result[date_col].dt.year - year_built
    result.loc[~result["BedBathRatio"].between(0, 20), "BedBathRatio"] = np.nan
    result.loc[~result["PropertyAge"].between(0, 300), "PropertyAge"] = np.nan

    return result

train_enriched = add_property_features(train_raw)
validation_enriched = add_property_features(validation_raw)
test_enriched = add_property_features(test_raw)

train_enriched[["BedBathRatio", "PropertyAge"]].describe().T

,count,mean,std,min,25%,50%,75%,max
BedBathRatio,129472.0,1.455147,0.472786,0.0,1.0,1.5,1.5,7.5
PropertyAge,129450.0,49.368227,27.628485,0.0,27.0,49.0,70.0,249.0


##Assign each property to a school district using spatial matching

In [9]:
districts = gpd.read_file(DISTRICT_PATH)

district_name_candidates = [
    "DistrictNa", "DistrictName", "DISTRICT", "NAME", "Name"
]
district_type_candidates = [
    "DistrictTy", "DistrictType", "TYPE", "Type"
]

district_name_col = next(
    (column for column in district_name_candidates if column in districts.columns),
    None
)
district_type_col = next(
    (column for column in district_type_candidates if column in districts.columns),
    None
)

if district_name_col is None:
    raise KeyError(
        "A school-district name field could not be detected. "
        f"Available fields: {districts.columns.tolist()}"
    )

keep_columns = [district_name_col, "geometry"]
if district_type_col is not None:
    keep_columns.insert(1, district_type_col)

districts = districts[keep_columns].copy()
districts = districts.loc[
    districts.geometry.notna() & ~districts.geometry.is_empty
].copy()

def spatially_attach_district(frame):
    result = frame.copy()
    result["_row_id"] = np.arange(len(result))

    longitude = pd.to_numeric(result["Longitude"], errors="coerce")
    latitude = pd.to_numeric(result["Latitude"], errors="coerce")

    valid_coordinates = (
        longitude.between(-125, -113) &
        latitude.between(32, 43)
    )

    valid_rows = result.loc[
        valid_coordinates,
        ["_row_id", "Longitude", "Latitude"]
    ].copy()

    points = gpd.GeoDataFrame(
        valid_rows,
        geometry=gpd.points_from_xy(
            pd.to_numeric(valid_rows["Longitude"], errors="coerce"),
            pd.to_numeric(valid_rows["Latitude"], errors="coerce")
        ),
        crs="EPSG:4326"
    ).to_crs(districts.crs)

    joined = gpd.sjoin(
        points,
        districts,
        how="left",
        predicate="within"
    )

    if district_type_col is not None:
        joined["_unified_priority"] = (
            joined[district_type_col]
            .astype(str)
            .str.contains("Unified", case=False, na=False)
            .astype(int)
        )
        joined = joined.sort_values(
            ["_row_id", "_unified_priority"],
            ascending=[True, False]
        )
    else:
        joined = joined.sort_values("_row_id")

    joined = joined.drop_duplicates("_row_id")
    district_map = joined.set_index("_row_id")[district_name_col]

    result["SchoolDistrictGeo"] = result["_row_id"].map(district_map)
    result["SchoolDistrictGeo"] = (
        result["SchoolDistrictGeo"]
        .fillna("Outside/Unmatched")
        .astype(str)
    )

    return result.drop(columns="_row_id")

train_enriched = spatially_attach_district(train_enriched)
validation_enriched = spatially_attach_district(validation_enriched)
test_enriched = spatially_attach_district(test_enriched)

district_coverage = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Matched Rows": [
        (train_enriched["SchoolDistrictGeo"] != "Outside/Unmatched").sum(),
        (validation_enriched["SchoolDistrictGeo"] != "Outside/Unmatched").sum(),
        (test_enriched["SchoolDistrictGeo"] != "Outside/Unmatched").sum()
    ],
    "Total Rows": [
        len(train_enriched),
        len(validation_enriched),
        len(test_enriched)
    ]
})

district_coverage["Matched %"] = (
    district_coverage["Matched Rows"] /
    district_coverage["Total Rows"] * 100
).round(2)

district_coverage

,Split,Matched Rows,Total Rows,Matched %
0,Train,129494,129555,99.95
1,Validation,43755,43782,99.94
2,Test,12854,12858,99.97


##Handle missing values and prepare the final modeling dataset

In [10]:
engineered_numeric_features = base_numeric_features + [
    "BedBathRatio",
    "PropertyAge"
]

updated_high_cardinality_features = base_high_cardinality_features + [
    "SchoolDistrictGeo"
]

all_numeric_features = engineered_numeric_features
all_categorical_features = (
    updated_high_cardinality_features + base_low_cardinality_features
)

for frame in [train_enriched, validation_enriched, test_enriched]:
    for column in all_numeric_features:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

train_medians = train_enriched[all_numeric_features].median()

for column in all_numeric_features:
    for frame in [train_enriched, validation_enriched, test_enriched]:
        frame[f"{column}_missing"] = frame[column].isna().astype("int8")
        frame[column] = frame[column].fillna(train_medians[column])

for column in all_categorical_features:
    for frame in [train_enriched, validation_enriched, test_enriched]:
        frame[column] = frame[column].fillna("Missing").astype(str)

for frame in [train_enriched, validation_enriched, test_enriched]:
    frame["PostalCode"] = (
        frame["PostalCode"]
        .str.replace(".0", "", regex=False)
        .str.strip()
    )

remaining_missing = {
    "Train": int(train_enriched[
        all_numeric_features + all_categorical_features
    ].isna().sum().sum()),
    "Validation": int(validation_enriched[
        all_numeric_features + all_categorical_features
    ].isna().sum().sum()),
    "Test": int(test_enriched[
        all_numeric_features + all_categorical_features
    ].isna().sum().sum())
}

remaining_missing

{'Train': 0, 'Validation': 0, 'Test': 0}

##Encode categorical features and construct model input matrices

In [11]:
def encode_feature_set(
    train_frame,
    validation_frame,
    test_frame,
    numeric_columns,
    high_cardinality_columns,
    low_cardinality_columns,
    encoding_target
):
    target_encoder = TargetEncoder(
        cols=high_cardinality_columns,
        smoothing=30,
        min_samples_leaf=20,
        handle_missing="value",
        handle_unknown="value",
        return_df=True
    )

    X_train_te = train_frame[high_cardinality_columns].reset_index(drop=True)
    y_train_te = encoding_target.reset_index(drop=True)

    train_high = target_encoder.fit_transform(
        X_train_te,
        y_train_te
    )
    validation_high = target_encoder.transform(
        validation_frame[high_cardinality_columns]
    )
    test_high = target_encoder.transform(
        test_frame[high_cardinality_columns]
    )

    renamed_high_columns = [
        f"{column}_target_encoded"
        for column in high_cardinality_columns
    ]
    train_high.columns = renamed_high_columns
    validation_high.columns = renamed_high_columns
    test_high.columns = renamed_high_columns

    onehot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        drop="first",
        sparse_output=False,
        dtype=np.float32
    )

    train_low_array = onehot_encoder.fit_transform(
        train_frame[low_cardinality_columns]
    )
    validation_low_array = onehot_encoder.transform(
        validation_frame[low_cardinality_columns]
    )
    test_low_array = onehot_encoder.transform(
        test_frame[low_cardinality_columns]
    )

    low_columns = onehot_encoder.get_feature_names_out(
        low_cardinality_columns
    )

    train_low = pd.DataFrame(
        train_low_array,
        columns=low_columns,
        index=train_frame.index
    )
    validation_low = pd.DataFrame(
        validation_low_array,
        columns=low_columns,
        index=validation_frame.index
    )
    test_low = pd.DataFrame(
        test_low_array,
        columns=low_columns,
        index=test_frame.index
    )

    missing_flag_columns = [
        f"{column}_missing"
        for column in numeric_columns
    ]
    direct_numeric_columns = numeric_columns + missing_flag_columns

    def combine(frame, high_frame, low_frame):
        combined = pd.concat(
            [
                frame[direct_numeric_columns].reset_index(drop=True),
                high_frame.reset_index(drop=True),
                low_frame.reset_index(drop=True)
            ],
            axis=1
        )

        return combined.astype(np.float32)

    X_train = combine(train_frame, train_high, train_low)
    X_validation = combine(validation_frame, validation_high, validation_low)
    X_test = combine(test_frame, test_high, test_low)

    X_validation = X_validation.reindex(
        columns=X_train.columns,
        fill_value=0
    )
    X_test = X_test.reindex(
        columns=X_train.columns,
        fill_value=0
    )

    return X_train, X_validation, X_test

y_train = train_enriched[target].reset_index(drop=True)
y_validation = validation_enriched[target].reset_index(drop=True)
y_test = test_enriched[target].reset_index(drop=True)

y_train_log = np.log1p(y_train)

X_train_original, X_validation_original, X_test_original = encode_feature_set(
    train_enriched,
    validation_enriched,
    test_enriched,
    numeric_columns=base_numeric_features,
    high_cardinality_columns=base_high_cardinality_features,
    low_cardinality_columns=base_low_cardinality_features,
    encoding_target=y_train_log
)

X_train_updated, X_validation_updated, X_test_updated = encode_feature_set(
    train_enriched,
    validation_enriched,
    test_enriched,
    numeric_columns=engineered_numeric_features,
    high_cardinality_columns=updated_high_cardinality_features,
    low_cardinality_columns=base_low_cardinality_features,
    encoding_target=y_train_log
)

matrix_summary = pd.DataFrame({
    "Feature Set": ["Original", "Updated"],
    "Train Rows": [
        X_train_original.shape[0],
        X_train_updated.shape[0]
    ],
    "Feature Count": [
        X_train_original.shape[1],
        X_train_updated.shape[1]
    ],
    "Contains BedBathRatio": [
        "BedBathRatio" in X_train_original.columns,
        "BedBathRatio" in X_train_updated.columns
    ],
    "Contains PropertyAge": [
        "PropertyAge" in X_train_original.columns,
        "PropertyAge" in X_train_updated.columns
    ],
    "Contains SchoolDistrictGeo": [
        any("SchoolDistrictGeo" in c for c in X_train_original.columns),
        any("SchoolDistrictGeo" in c for c in X_train_updated.columns)
    ]
})

matrix_summary

,Feature Set,Train Rows,Feature Count,Contains BedBathRatio,Contains PropertyAge,Contains SchoolDistrictGeo
0,Original,129555,37,False,False,False
1,Updated,129555,42,True,True,True


##Export the processed datasets for future use

In [12]:
def export_split(X, source_frame, split_name, file_name):
    export_frame = X.copy()
    export_frame[target] = source_frame[target].reset_index(drop=True)
    export_frame[date_col] = source_frame[date_col].reset_index(drop=True)
    export_frame["split"] = split_name

    output_file = os.path.join(SPLIT_PATH, file_name)
    export_frame.to_csv(output_file, index=False)
    return output_file

exported_files = {
    "Original Train": export_split(
        X_train_original, train_enriched, "train",
        "cleaned_train_original_week6.csv"
    ),
    "Original Validation": export_split(
        X_validation_original, validation_enriched, "validation",
        "cleaned_validation_original_week6.csv"
    ),
    "Original Test": export_split(
        X_test_original, test_enriched, "test",
        "cleaned_test_original_week6.csv"
    ),
    "Updated Train": export_split(
        X_train_updated, train_enriched, "train",
        "cleaned_train_updated_week6.csv"
    ),
    "Updated Validation": export_split(
        X_validation_updated, validation_enriched, "validation",
        "cleaned_validation_updated_week6.csv"
    ),
    "Updated Test": export_split(
        X_test_updated, test_enriched, "test",
        "cleaned_test_updated_week6.csv"
    )
}

pd.Series(exported_files, name="Saved File")

,Saved File
Original Train,/content/drive/MyDrive/2026-Summer/IDXExchange...
Original Validation,/content/drive/MyDrive/2026-Summer/IDXExchange...
Original Test,/content/drive/MyDrive/2026-Summer/IDXExchange...
Updated Train,/content/drive/MyDrive/2026-Summer/IDXExchange...
Updated Validation,/content/drive/MyDrive/2026-Summer/IDXExchange...
Updated Test,/content/drive/MyDrive/2026-Summer/IDXExchange...


##Define evaluation metrics and candidate machine learning models

In [13]:
def dollar_predictions_from_log(model, X):
    prediction_log = model.predict(X)
    prediction_dollars = np.expm1(prediction_log)

    return np.maximum(prediction_dollars, 0)

def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100
    }

candidate_models = {
    "Log Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Log HistGradientBoosting": HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=350,
        max_leaf_nodes=31,
        min_samples_leaf=30,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    ),
    "Log Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        max_features=0.85,
        min_samples_leaf=2,
        max_depth=None,
        n_jobs=-1,
        random_state=42
    ),
    "Log Random Forest": RandomForestRegressor(
        n_estimators=100,
        max_features=0.75,
        min_samples_leaf=2,
        max_depth=None,
        n_jobs=-1,
        random_state=42
    )
}

feature_sets = {
    "Original": (
        X_train_original,
        X_validation_original,
        X_test_original
    ),
    "Updated": (
        X_train_updated,
        X_validation_updated,
        X_test_updated
    )
}

##Train each model and evaluate validation performance

In [14]:
validation_rows = []
fitted_models = {}
validation_predictions = {}

for feature_set_name, (X_train_set, X_validation_set, _) in feature_sets.items():
    print(f"\n===== {feature_set_name} =====", flush=True)

    for model_name, model_template in candidate_models.items():
        print(f"Training {model_name}...", flush=True)

        model = clone(model_template)

        start_time = time.perf_counter()
        model.fit(X_train_set, y_train_log)
        elapsed_seconds = time.perf_counter() - start_time

        print(f"Done in {elapsed_seconds:.1f} sec", flush=True)

        validation_pred = dollar_predictions_from_log(model, X_validation_set)
        metrics = regression_metrics(y_validation, validation_pred)

        validation_rows.append({
            "Feature Set": feature_set_name,
            "Model": model_name,
            "Feature Count": X_train_set.shape[1],
            "Training Seconds": elapsed_seconds,
            **metrics
        })

        fitted_models[(feature_set_name, model_name)] = model
        validation_predictions[(feature_set_name, model_name)] = validation_pred

validation_results = pd.DataFrame(validation_rows).sort_values(
    ["R2", "RMSE", "MAE"],
    ascending=[False, True, True]
).reset_index(drop=True)

validation_results


===== Original =====
Training Log Linear Regression...
Done in 0.2 sec
Training Log HistGradientBoosting...
Done in 14.2 sec
Training Log Extra Trees...
Done in 79.8 sec
Training Log Random Forest...
Done in 158.9 sec

===== Updated =====
Training Log Linear Regression...
Done in 0.2 sec
Training Log HistGradientBoosting...
Done in 14.1 sec
Training Log Extra Trees...
Done in 89.4 sec
Training Log Random Forest...
Done in 174.9 sec


,Feature Set,Model,Feature Count,Training Seconds,R2,RMSE,MAE,MAPE
0,Original,Log HistGradientBoosting,37,14.236597,0.023165,8.236692e+06,294609.076257,16.765943
1,Original,Log Random Forest,37,158.916561,0.023080,8.237049e+06,289118.566985,16.392414
2,Updated,Log HistGradientBoosting,42,14.104160,0.023078,8.237057e+06,293792.987601,16.678838
3,Updated,Log Random Forest,42,174.869654,0.023027,8.237271e+06,287864.737672,16.309496
4,Updated,Log Extra Trees,42,89.397787,0.022863,8.237966e+06,291540.903798,16.598550
5,Original,Log Extra Trees,37,79.777579,0.022856,8.237992e+06,292558.734868,16.688476
6,Updated,Log Linear Regression,42,0.203184,-0.036151,8.483080e+06,379607.620316,21.709803
7,Original,Log Linear Regression,37,0.237110,-0.040314,8.500108e+06,384541.420768,22.203738


##Compare the original and engineered feature sets

In [15]:
model_comparison = validation_results.pivot(
    index="Model",
    columns="Feature Set",
    values=["R2", "RMSE", "MAE", "MAPE"]
)

delta_rows = []

for model_name in validation_results["Model"].unique():
    original_row = validation_results.loc[
        (validation_results["Model"] == model_name) &
        (validation_results["Feature Set"] == "Original")
    ].iloc[0]

    updated_row = validation_results.loc[
        (validation_results["Model"] == model_name) &
        (validation_results["Feature Set"] == "Updated")
    ].iloc[0]

    delta_rows.append({
        "Model": model_name,
        "Original R2": original_row["R2"],
        "Updated R2": updated_row["R2"],
        "R2 Change": updated_row["R2"] - original_row["R2"],
        "Original RMSE": original_row["RMSE"],
        "Updated RMSE": updated_row["RMSE"],
        "RMSE Change": updated_row["RMSE"] - original_row["RMSE"],
        "Updated Features Improved R2": (
            updated_row["R2"] > original_row["R2"]
        )
    })

feature_delta = (
    pd.DataFrame(delta_rows)
    .sort_values("R2 Change", ascending=False)
    .reset_index(drop=True)
)

feature_delta

,Model,Original R2,Updated R2,R2 Change,Original RMSE,Updated RMSE,RMSE Change,Updated Features Improved R2
0,Log Linear Regression,-0.040314,-0.036151,0.004164,8.500108e+06,8.483080e+06,-17027.035173,True
1,Log Extra Trees,0.022856,0.022863,0.000006,8.237992e+06,8.237966e+06,-25.559555,True
2,Log Random Forest,0.023080,0.023027,-0.000053,8.237049e+06,8.237271e+06,222.034486,False
3,Log HistGradientBoosting,0.023165,0.023078,-0.000086,8.236692e+06,8.237057e+06,364.579960,False


##Select the best-performing validation model

In [16]:
best_validation_row = validation_results.iloc[0]

selected_feature_set = best_validation_row["Feature Set"]
selected_model_name = best_validation_row["Model"]

selected_model = fitted_models[
    (selected_feature_set, selected_model_name)
]
selected_validation_predictions = validation_predictions[
    (selected_feature_set, selected_model_name)
]

print("Selected feature set:", selected_feature_set)
print("Selected model:", selected_model_name)
print("Validation R²:", round(best_validation_row["R2"], 4))
print("Validation RMSE:", f"${best_validation_row['RMSE']:,.0f}")
print("Validation MAE:", f"${best_validation_row['MAE']:,.0f}")

Selected feature set: Original
Selected model: Log HistGradientBoosting
Validation R²: 0.0232
Validation RMSE: $8,236,692
Validation MAE: $294,609


##Retrain the selected model and evaluate it on the test set

In [17]:
X_train_selected, X_validation_selected, X_test_selected = feature_sets[
    selected_feature_set
]

X_final = pd.concat(
    [X_train_selected, X_validation_selected],
    ignore_index=True
)
y_final = pd.concat(
    [y_train, y_validation],
    ignore_index=True
)
y_final_log = np.log1p(y_final)

final_model = clone(candidate_models[selected_model_name])
final_model.fit(X_final, y_final_log)

test_predictions = dollar_predictions_from_log(
    final_model,
    X_test_selected
)
test_metrics = regression_metrics(
    y_test,
    test_predictions
)

final_summary = pd.DataFrame([{
    "Selected Feature Set": selected_feature_set,
    "Selected Model": selected_model_name,
    "Validation R2": best_validation_row["R2"],
    "Validation RMSE": best_validation_row["RMSE"],
    "Validation MAE": best_validation_row["MAE"],
    "Validation MAPE": best_validation_row["MAPE"],
    "Test R2": test_metrics["R2"],
    "Test RMSE": test_metrics["RMSE"],
    "Test MAE": test_metrics["MAE"],
    "Test MAPE": test_metrics["MAPE"]
}])

final_summary

,Selected Feature Set,Selected Model,Validation R2,Validation RMSE,Validation MAE,Validation MAPE,Test R2,Test RMSE,Test MAE,Test MAPE
0,Original,Log HistGradientBoosting,0.023165,8.236692e+06,294609.076257,16.765943,0.801158,685108.389307,200634.036264,20.619517


##Display the most important features used by the selected model

In [18]:
selected_feature_names = X_final.columns

if selected_model_name == "Log Linear Regression":
    model_step = final_model.named_steps["model"]
    importance_values = np.abs(model_step.coef_)
    importance_label = "Absolute Standardized Coefficient"
else:
    if hasattr(final_model, "feature_importances_"):
        importance_values = final_model.feature_importances_
        importance_label = "Feature Importance"
    else:
        importance_values = None
        importance_label = "Importance"

if importance_values is not None:
    feature_importance = (
        pd.DataFrame({
            "Feature": selected_feature_names,
            importance_label: importance_values
        })
        .sort_values(importance_label, ascending=False)
        .reset_index(drop=True)
    )

    top_feature_importance = feature_importance.head(20).sort_values(
        importance_label
    )

    plt.figure(figsize=(10, 7))
    plt.barh(
        top_feature_importance["Feature"],
        top_feature_importance[importance_label]
    )
    plt.xlabel(importance_label)
    plt.title(
        f"Top 20 Features — {selected_feature_set} / "
        f"{selected_model_name}"
    )
    plt.tight_layout()
    plt.show()

    display(feature_importance.head(20))
else:
    feature_importance = pd.DataFrame(
        columns=["Feature", importance_label]
    )
    print(
        "The selected model does not expose native feature importance."
    )

The selected model does not expose native feature importance.


##Summarize the impact of the engineered features.

In [19]:
best_original = (
    validation_results.loc[
        validation_results["Feature Set"] == "Original"
    ]
    .sort_values(
        ["R2", "RMSE", "MAE"],
        ascending=[False, True, True]
    )
    .iloc[0]
)

best_updated = (
    validation_results.loc[
        validation_results["Feature Set"] == "Updated"
    ]
    .sort_values(
        ["R2", "RMSE", "MAE"],
        ascending=[False, True, True]
    )
    .iloc[0]
)

r2_difference = best_updated["R2"] - best_original["R2"]
rmse_difference = best_updated["RMSE"] - best_original["RMSE"]

if r2_difference > 0 and rmse_difference < 0:
    feature_conclusion = (
        "The Week 6 feature set improved both validation R² and RMSE."
    )
elif r2_difference > 0:
    feature_conclusion = (
        "The Week 6 feature set improved validation R², although not "
        "every error metric improved."
    )
elif rmse_difference < 0:
    feature_conclusion = (
        "The Week 6 feature set reduced validation RMSE, although R² "
        "did not improve."
    )
else:
    feature_conclusion = (
        "The Week 6 features did not improve the best validation result "
        "in this experiment. This is still a valid finding."
    )

interpretation = pd.DataFrame({
    "Item": [
        "Best original model",
        "Best updated model",
        "Best overall validation pipeline",
        "Updated minus original R2",
        "Updated minus original RMSE",
        "Interpretation"
    ],
    "Result": [
        best_original["Model"],
        best_updated["Model"],
        f"{selected_feature_set} — {selected_model_name}",
        f"{r2_difference:.4f}",
        f"${rmse_difference:,.0f}",
        feature_conclusion
    ]
})

interpretation

,Item,Result
0,Best original model,Log HistGradientBoosting
1,Best updated model,Log HistGradientBoosting
2,Best overall validation pipeline,Original — Log HistGradientBoosting
3,Updated minus original R2,-0.0001
4,Updated minus original RMSE,$365
5,Interpretation,The Week 6 features did not improve the best v...


# Continue

In [20]:
# Tune HistGradientBoosting and compare parameter combinations on the validation set.

hgb_tuning_rows = []

hgb_parameter_sets = [
    {
        "learning_rate": 0.03,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20
    },
    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 20
    },
    {
        "learning_rate": 0.05,
        "max_iter": 400,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 30
    },
    {
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 63,
        "min_samples_leaf": 20
    },
    {
        "learning_rate": 0.10,
        "max_iter": 250,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 30
    }
]

for number, parameters in enumerate(hgb_parameter_sets, start=1):
    print(
        f"Training HistGradientBoosting {number}/"
        f"{len(hgb_parameter_sets)}...",
        flush=True
    )

    model = HistGradientBoostingRegressor(
        **parameters,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    )

    start_time = time.perf_counter()

    model.fit(
        X_train_original,
        y_train_log
    )

    training_seconds = time.perf_counter() - start_time

    validation_predictions_hgb = dollar_predictions_from_log(
        model,
        X_validation_original
    )

    metrics = regression_metrics(
        y_validation,
        validation_predictions_hgb
    )

    hgb_tuning_rows.append({
        **parameters,
        "Training Seconds": training_seconds,
        **metrics
    })

hgb_tuning_results = (
    pd.DataFrame(hgb_tuning_rows)
    .sort_values(
        ["R2", "RMSE", "MAE"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

hgb_tuning_results

Training HistGradientBoosting 1/5...
Training HistGradientBoosting 2/5...
Training HistGradientBoosting 3/5...
Training HistGradientBoosting 4/5...
Training HistGradientBoosting 5/5...


,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,Training Seconds,R2,RMSE,MAE,MAPE
0,0.05,300,63,20,14.609850,0.023444,8.235516e+06,288935.887581,16.342994
1,0.10,250,31,30,8.533968,0.023334,8.235979e+06,290910.601760,16.449577
2,0.05,400,31,30,14.509107,0.023242,8.236368e+06,292787.887691,16.621751
3,0.05,300,31,20,11.428715,0.023041,8.237214e+06,297105.079719,16.906072
4,0.03,300,31,20,12.565540,0.022704,8.238633e+06,306557.728224,17.405140


In [21]:
# Tune Random Forest using a small number of parameter combinations.

rf_tuning_rows = []

rf_parameter_sets = [
    {
        "n_estimators": 100,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": 0.75
    },
    {
        "n_estimators": 150,
        "max_depth": 25,
        "min_samples_leaf": 2,
        "max_features": 0.75
    },
    {
        "n_estimators": 150,
        "max_depth": None,
        "min_samples_leaf": 3,
        "max_features": 0.60
    },
    {
        "n_estimators": 200,
        "max_depth": 20,
        "min_samples_leaf": 3,
        "max_features": 0.85
    }
]

for number, parameters in enumerate(rf_parameter_sets, start=1):
    print(
        f"Training Random Forest {number}/"
        f"{len(rf_parameter_sets)}...",
        flush=True
    )

    model = RandomForestRegressor(
        **parameters,
        n_jobs=-1,
        random_state=42
    )

    start_time = time.perf_counter()

    model.fit(
        X_train_original,
        y_train_log
    )

    training_seconds = time.perf_counter() - start_time

    validation_predictions_rf = dollar_predictions_from_log(
        model,
        X_validation_original
    )

    metrics = regression_metrics(
        y_validation,
        validation_predictions_rf
    )

    rf_tuning_rows.append({
        **parameters,
        "Training Seconds": training_seconds,
        **metrics
    })

rf_tuning_results = (
    pd.DataFrame(rf_tuning_rows)
    .sort_values(
        ["R2", "RMSE", "MAE"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

rf_tuning_results

Training Random Forest 1/4...
Training Random Forest 2/4...
Training Random Forest 3/4...
Training Random Forest 4/4...


,n_estimators,max_depth,min_samples_leaf,max_features,Training Seconds,R2,RMSE,MAE,MAPE
0,150,25.0,2,0.75,235.543224,0.023138,8.236806e+06,288451.813125,16.399693
1,150,NaN,3,0.60,178.519795,0.023075,8.237068e+06,288418.653011,16.392621
2,100,20.0,2,0.75,142.195704,0.023055,8.237156e+06,289360.475021,16.424124
3,200,20.0,3,0.85,311.787262,0.022954,8.237579e+06,290038.806855,16.438388


In [22]:
# Compare the contribution of each engineered feature.

feature_selection_sets = {
    "Original": {
        "numeric": base_numeric_features,
        "high_cardinality": base_high_cardinality_features
    },

    "Original + BedBathRatio": {
        "numeric": base_numeric_features + ["BedBathRatio"],
        "high_cardinality": base_high_cardinality_features
    },

    "Original + PropertyAge": {
        "numeric": base_numeric_features + ["PropertyAge"],
        "high_cardinality": base_high_cardinality_features
    },

    "Original + SchoolDistrictGeo": {
        "numeric": base_numeric_features,
        "high_cardinality": (
            base_high_cardinality_features + ["SchoolDistrictGeo"]
        )
    },

    "Original + All Engineered Features": {
        "numeric": (
            base_numeric_features +
            ["BedBathRatio", "PropertyAge"]
        ),
        "high_cardinality": (
            base_high_cardinality_features +
            ["SchoolDistrictGeo"]
        )
    }
}

feature_selection_matrices = {}
feature_selection_rows = []

for feature_set_name, feature_definition in feature_selection_sets.items():
    print(f"Preparing {feature_set_name}...", flush=True)

    X_train_subset, X_validation_subset, X_test_subset = encode_feature_set(
        train_enriched,
        validation_enriched,
        test_enriched,
        numeric_columns=feature_definition["numeric"],
        high_cardinality_columns=feature_definition["high_cardinality"],
        low_cardinality_columns=base_low_cardinality_features,
        encoding_target=y_train_log
    )

    feature_selection_matrices[feature_set_name] = (
        X_train_subset,
        X_validation_subset,
        X_test_subset
    )

    feature_model = HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    )

    feature_model.fit(
        X_train_subset,
        y_train_log
    )

    feature_predictions = dollar_predictions_from_log(
        feature_model,
        X_validation_subset
    )

    metrics = regression_metrics(
        y_validation,
        feature_predictions
    )

    feature_selection_rows.append({
        "Feature Set": feature_set_name,
        "Feature Count": X_train_subset.shape[1],
        **metrics
    })

feature_selection_results = (
    pd.DataFrame(feature_selection_rows)
    .sort_values(
        ["R2", "RMSE", "MAE"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

feature_selection_results

Preparing Original...
Preparing Original + BedBathRatio...
Preparing Original + PropertyAge...
Preparing Original + SchoolDistrictGeo...
Preparing Original + All Engineered Features...


,Feature Set,Feature Count,R2,RMSE,MAE,MAPE
0,Original + BedBathRatio,39,0.023176,8.236647e+06,296922.500305,16.862482
1,Original + All Engineered Features,42,0.023161,8.236707e+06,296050.634135,16.810039
2,Original + SchoolDistrictGeo,38,0.023083,8.237036e+06,295507.853585,16.753630
3,Original,37,0.023041,8.237214e+06,297105.079719,16.906072
4,Original + PropertyAge,39,0.023037,8.237231e+06,297484.026479,16.897167


In [23]:
# Compare the original baseline models with the best tuned models.

best_hgb_parameters = {
    "learning_rate": float(
        hgb_tuning_results.iloc[0]["learning_rate"]
    ),
    "max_iter": int(
        hgb_tuning_results.iloc[0]["max_iter"]
    ),
    "max_leaf_nodes": int(
        hgb_tuning_results.iloc[0]["max_leaf_nodes"]
    ),
    "min_samples_leaf": int(
        hgb_tuning_results.iloc[0]["min_samples_leaf"]
    )
}

best_rf_parameters = {
    "n_estimators": int(
        rf_tuning_results.iloc[0]["n_estimators"]
    ),
    "max_depth": (
        None
        if pd.isna(rf_tuning_results.iloc[0]["max_depth"])
        else int(rf_tuning_results.iloc[0]["max_depth"])
    ),
    "min_samples_leaf": int(
        rf_tuning_results.iloc[0]["min_samples_leaf"]
    ),
    "max_features": float(
        rf_tuning_results.iloc[0]["max_features"]
    )
}

best_feature_set_name = feature_selection_results.iloc[0]["Feature Set"]

X_train_week7, X_validation_week7, X_test_week7 = (
    feature_selection_matrices[best_feature_set_name]
)

tuned_models = {
    "Tuned HistGradientBoosting": HistGradientBoostingRegressor(
        **best_hgb_parameters,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    ),

    "Tuned Random Forest": RandomForestRegressor(
        **best_rf_parameters,
        n_jobs=-1,
        random_state=42
    )
}

tuned_comparison_rows = []
tuned_fitted_models = {}

for model_name, model in tuned_models.items():
    print(f"Training {model_name}...", flush=True)

    start_time = time.perf_counter()

    model.fit(
        X_train_week7,
        y_train_log
    )

    training_seconds = time.perf_counter() - start_time

    validation_predictions_tuned = dollar_predictions_from_log(
        model,
        X_validation_week7
    )

    metrics = regression_metrics(
        y_validation,
        validation_predictions_tuned
    )

    tuned_comparison_rows.append({
        "Model": model_name,
        "Feature Set": best_feature_set_name,
        "Tuned": True,
        "Training Seconds": training_seconds,
        **metrics
    })

    tuned_fitted_models[model_name] = model

tuned_model_results = (
    pd.DataFrame(tuned_comparison_rows)
    .sort_values(
        ["R2", "RMSE", "MAE"],
        ascending=[False, True, True]
    )
    .reset_index(drop=True)
)

baseline_for_comparison = (
    validation_results[
        validation_results["Feature Set"] == "Original"
    ][
        [
            "Model",
            "Feature Set",
            "Training Seconds",
            "R2",
            "RMSE",
            "MAE",
            "MAPE"
        ]
    ]
    .copy()
)

baseline_for_comparison["Tuned"] = False

week7_model_comparison = pd.concat(
    [
        baseline_for_comparison,
        tuned_model_results
    ],
    ignore_index=True
).sort_values(
    ["R2", "RMSE", "MAE"],
    ascending=[False, True, True]
).reset_index(drop=True)

week7_model_comparison

Training Tuned HistGradientBoosting...
Training Tuned Random Forest...


,Model,Feature Set,Training Seconds,R2,RMSE,MAE,MAPE,Tuned
0,Tuned HistGradientBoosting,Original + BedBathRatio,15.256598,0.023426,8.235591e+06,289335.620050,16.386302,True
1,Log HistGradientBoosting,Original,14.236597,0.023165,8.236692e+06,294609.076257,16.765943,False
2,Tuned Random Forest,Original + BedBathRatio,240.797899,0.023089,8.237013e+06,288853.120047,16.432317,True
3,Log Random Forest,Original,158.916561,0.023080,8.237049e+06,289118.566985,16.392414,False
4,Log Extra Trees,Original,79.777579,0.022856,8.237992e+06,292558.734868,16.688476,False
5,Log Linear Regression,Original,0.237110,-0.040314,8.500108e+06,384541.420768,22.203738,False


In [24]:
# Select the best tuned model based on validation performance.

best_tuned_row = tuned_model_results.iloc[0]

best_tuned_model_name = best_tuned_row["Model"]
best_tuned_model = tuned_fitted_models[best_tuned_model_name]

print("Best feature set:", best_feature_set_name)
print("Best tuned model:", best_tuned_model_name)
print("Validation R²:", round(best_tuned_row["R2"], 4))
print("Validation RMSE:", f"${best_tuned_row['RMSE']:,.0f}")
print("Validation MAE:", f"${best_tuned_row['MAE']:,.0f}")
print("Validation MAPE:", f"{best_tuned_row['MAPE']:.2f}%")

Best feature set: Original + BedBathRatio
Best tuned model: Tuned HistGradientBoosting
Validation R²: 0.0234
Validation RMSE: $8,235,591
Validation MAE: $289,336
Validation MAPE: 16.39%


In [25]:
# Retrain the selected tuned model and evaluate it on the final test set.

X_train_validation_week7 = pd.concat(
    [
        X_train_week7,
        X_validation_week7
    ],
    ignore_index=True
)

y_train_validation_week7 = pd.concat(
    [
        y_train,
        y_validation
    ],
    ignore_index=True
)

y_train_validation_week7_log = np.log1p(
    y_train_validation_week7
)

if best_tuned_model_name == "Tuned HistGradientBoosting":
    final_week7_model = HistGradientBoostingRegressor(
        **best_hgb_parameters,
        l2_regularization=1.0,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42
    )
else:
    final_week7_model = RandomForestRegressor(
        **best_rf_parameters,
        n_jobs=-1,
        random_state=42
    )

final_week7_model.fit(
    X_train_validation_week7,
    y_train_validation_week7_log
)

week7_test_predictions = dollar_predictions_from_log(
    final_week7_model,
    X_test_week7
)

week7_test_metrics = regression_metrics(
    y_test,
    week7_test_predictions
)

week7_final_summary = pd.DataFrame([{
    "Selected Feature Set": best_feature_set_name,
    "Selected Model": best_tuned_model_name,
    "Validation R2": best_tuned_row["R2"],
    "Validation RMSE": best_tuned_row["RMSE"],
    "Validation MAE": best_tuned_row["MAE"],
    "Test R2": week7_test_metrics["R2"],
    "Test RMSE": week7_test_metrics["RMSE"],
    "Test MAE": week7_test_metrics["MAE"],
    "Test MAPE": week7_test_metrics["MAPE"]
}])

week7_final_summary

,Selected Feature Set,Selected Model,Validation R2,Validation RMSE,Validation MAE,Test R2,Test RMSE,Test MAE,Test MAPE
0,Original + BedBathRatio,Tuned HistGradientBoosting,0.023426,8.235591e+06,289335.62005,0.821673,648803.613834,193948.977637,20.243703


In [26]:
# Summarize the Week 7 tuning and feature selection results.

baseline_best_row = validation_results.iloc[0]

validation_r2_change = (
    best_tuned_row["R2"] -
    baseline_best_row["R2"]
)

validation_rmse_change = (
    best_tuned_row["RMSE"] -
    baseline_best_row["RMSE"]
)

if validation_r2_change > 0 and validation_rmse_change < 0:
    tuning_conclusion = (
        "Hyperparameter tuning improved both validation R² and RMSE."
    )
elif validation_r2_change > 0:
    tuning_conclusion = (
        "Hyperparameter tuning improved validation R², "
        "although RMSE did not improve."
    )
elif validation_rmse_change < 0:
    tuning_conclusion = (
        "Hyperparameter tuning reduced validation RMSE, "
        "although R² did not improve."
    )
else:
    tuning_conclusion = (
        "The tested tuning combinations did not improve the "
        "best baseline validation performance."
    )

week7_interpretation = pd.DataFrame({
    "Item": [
        "Best baseline model",
        "Best tuned model",
        "Selected feature set",
        "Validation R2 change",
        "Validation RMSE change",
        "Conclusion"
    ],
    "Result": [
        baseline_best_row["Model"],
        best_tuned_model_name,
        best_feature_set_name,
        f"{validation_r2_change:.4f}",
        f"${validation_rmse_change:,.0f}",
        tuning_conclusion
    ]
})

week7_interpretation

,Item,Result
0,Best baseline model,Log HistGradientBoosting
1,Best tuned model,Tuned HistGradientBoosting
2,Selected feature set,Original + BedBathRatio
3,Validation R2 change,0.0003
4,Validation RMSE change,"$-1,101"
5,Conclusion,Hyperparameter tuning improved both validation...
